In [1]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


## Create Dataset

In [2]:
from sklearn.model_selection import train_test_split
import pandas as pd
import random
import os

def create_multiclass_dataset(label_dict, train_test_ratio=0.8, seed=42, 
                              ds_file="dataset.csv"):
    """
    Creates a multiclass dataset from multiple binary labeled files, shuffles it, and splits into train/test sets.
    
    Parameters:
    - label_dict (dict): A dictionary where the keys are class labels (int) and the values are lists of file paths (str).
    - train_test_ratio (float): The ratio of the training set size to the total dataset (between 0 and 1).
    - seed (int): Seed for shuffling the dataset to ensure reproducibility.
    - train_file (str): The path to save the resulting training dataset CSV file.
    - test_file (str): The path to save the resulting test dataset CSV file.
    
    Returns:
    - DataFrame, DataFrame: The training and test datasets as pandas DataFrames.
    """
    data = []  # List to store all rows of data

    # Iterate through the dictionary of class labels and file paths
    for class_label, file_paths in label_dict.items():
        for file_path in file_paths:
            with open(file_path, 'r') as file:
                for line in file:
                    # Split the line by tab to get the binary label and the string
                    parts = line.strip().split('\t')
                    if len(parts) != 2:
                        continue  # Skip malformed lines
                    binary_label, string = parts
                    if binary_label == '0':
                        # Assign "Unknown" to binary label 0
                        data.append(['Unknown', string])
                    elif binary_label == '1':
                        # Assign the corresponding class label for binary label 1
                        data.append([class_label, string])

    #Filter out some of the Unknown labels
    unknown_samples = [sample for sample in data if sample[0] == 'Unknown']
    subset_size = 10000  # Size of the random subset you want to delete
    subset_to_delete = random.sample(unknown_samples, subset_size)
    data = [sample for sample in data if sample not in subset_to_delete]

    #Turn T's to U's
    data = [[sample[0], sample[1].replace("T", "U")] for sample in data]
    data = [[sample[0], sample[1].replace("t", "u")] for sample in data]
    
    # Create a pandas DataFrame from the data
    df = pd.DataFrame(data, columns=['Label', 'String'])
    df.to_csv(ds_file, index=False)
    # Shuffle the DataFrame with the provided seed for reproducibility
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    # Split the dataset into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(df['String'], df['Label'], test_size=0.2, random_state=seed)

    # Return the training and test DataFrames
    return  X_train, X_test, y_train, y_test

In [3]:
# Example usage
label_dict = {
    "hnRNPL": ["//home//omri//RBPMap//data//hnRNPL//ENCSR724RDN//hnRNPL_ENCSR724RDN_dataset.txt", "//home//omri//RBPMap//data//hnRNPL//ENCSR795CAI//hnRNPL_ENCSR795CAI_dataset.txt"],
    "PUM2": ["//home//omri//RBPMap//data//PUM2//ENCSR661ICQ//PUM2_ENCSR661ICQ_dataset.txt"],
    "QKI": ["//home//omri//RBPMap//data//QKI//ENCSR366YOG//QKI_ENCSR366YOG_dataset.txt", "//home//omri//RBPMap//data//QKI//ENCSR570WLM//QKI_ENCSR570WLM_dataset.txt"],
    "RBM5": ["//home//omri//RBPMap//data//RBM5//ENCSR489ABS//RBM5_ENCSR489ABS_dataset.txt"],
    "SRSF1": ["//home//omri//RBPMap//data//SRSF1//ENCSR321PWZ//SRSF1_ENCSR321PWZ_dataset.txt", "//home//omri//RBPMap//data//SRSF1//ENCSR432XUP//SRSF1_ENCSR432XUP_dataset.txt"],
}
seed = 42  # Example seed for shuffling
train_test_ratio = 0.8  # 80% for training, 20% for testing

# Create the multiclass dataset, shuffle, split, and save as train/test CSV files
X_train, X_test, y_train, y_test = create_multiclass_dataset(label_dict, train_test_ratio, seed, ds_file="dataset.csv")

# Check the first few rows of the train and test datasets
# print(X_train.info())
# print("Training Set:\n", X_train.head())
# print(X_test.info())
# print("Test Set:\n", X_test.head())
# print(y_train.info())
# print(y_train.head())
# print(y_test.info())
# print(y_test.head())

In [4]:
from RNADataset import RNADataset
from torch.utils.data import DataLoader
# Create the dataset
train_ds = RNADataset(X_train, y_train, device)
test_ds = RNADataset(X_test, y_test, device)

# Create the DataLoader
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=True)


{0, 1, 2, 3, 4, 5}
{0, 1, 2, 3, 4, 5}


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from multimolecule import RnaTokenizer, RnaBertForSequencePrediction
from torch.optim import AdamW

# Load the RNABERT tokenizer and model
model_name = "multimolecule/rnabert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6).to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

/home/omri/miniconda3/envs/RBP/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of RnaBertForSequencePrediction were not initialized from the model checkpoint at multimolecule/rnabert and are newly initialized: ['sequence_head.decoder.bias', 'sequence_head.decoder.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
num_epochs = 100
model.train()  # Set the model to training mode

for epoch in range(num_epochs):
    for seqs, labels in train_dl:
        # Tokenize the inputs
        inputs = tokenizer(list(seqs), padding=True, truncation=True, return_tensors='pt').to(device)

        # Move tensors to the device (GPU/CPU)
        input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']
        # labels = labels.to(input_ids.device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss  # Get the loss from the model output

        # Backward pass
        optimizer.zero_grad()  # Clear previous gradients
        loss.backward()  # Compute gradients
        optimizer.step()  # Update model weights

        print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')  # Print loss for this batch


/opt/conda/conda-bld/pytorch_1720538459595/work/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [0,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1720538459595/work/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [3,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1720538459595/work/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [4,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1720538459595/work/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [5,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1720538459595/work/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [6,0,0] Assertion `t >= 0 && t < n_cl

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
